# Ponds1.csv — Extensive Descriptive Statistics

This notebook provides a thorough statistical and visual exploration of **Ponds1.csv**, a pond water-quality dataset spanning nearly a full year (Feb 2022 – Jan 2023) across three stations. The primary welfare indicator is **Dissolved Oxygen (DO)**: values below 3 mg/L are considered acutely stressful for fish. The analysis mirrors the structure of `01_ponds_descriptive_stats.ipynb` (which covers Ponds.csv, Feb–Apr 2022) but extends it substantially to leverage the longer time span and richer seasonal variation.

## 0. Setup

In [1]:
import os
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from scipy import stats

sns.set_theme(style='whitegrid', palette='tab10')
plt.rcParams.update({'figure.dpi': 130, 'font.size': 10})

DATA_PATH  = '../data/kaggle/Ponds1.csv'
SAVE_DIR   = 'descriptives'
DO_THRESHOLD = 3.0

os.makedirs(SAVE_DIR, exist_ok=True)
print('Setup complete. Save dir:', os.path.abspath(SAVE_DIR))

Setup complete. Save dir: /Users/dlau/repos/fish-welfare/transfer_kaggle/descriptives


## 1. Load & Parse

We read the raw CSV, normalise station labels to lowercase, coerce Excel error strings (`#VALUE!`) to NaN, combine the separate Date and Time columns into a single datetime index, and engineer several derived time features used throughout the analysis.

In [2]:
df = pd.read_csv(DATA_PATH, low_memory=False)

# Normalise station names
df['station'] = df['station'].str.lower().str.strip()

# Coerce error strings to NaN for numeric columns
numeric_cols = ['NITRATE(PPM)', 'PH', 'AMMONIA(mg/l)', 'TEMP', 'DO', 'TURBIDITY', 'MANGANESE(mg/l)']
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Combine Date + Time → datetime
df['datetime'] = pd.to_datetime(
    df['Date'].astype(str) + ' ' + df['Time'].astype(str),
    dayfirst=True
)

df = df.sort_values(['station', 'datetime']).reset_index(drop=True)

# Derived time features
df['date_only']  = df['datetime'].dt.date
df['month']      = df['datetime'].dt.month
df['month_name'] = df['datetime'].dt.strftime('%b')
df['hour']       = df['datetime'].dt.hour
df['dow']        = df['datetime'].dt.dayofweek
df['week']       = df['datetime'].dt.isocalendar().week.astype('Int64').astype(float).astype('Int64')

def get_season(m):
    if m in [12, 1, 2]:  return 'DJF'
    elif m in [3, 4, 5]: return 'MAM'
    elif m in [6, 7, 8]: return 'JJA'
    else:                return 'SON'

df['season']  = df['month'].apply(get_season)
df['low_DO']  = df['DO'] < DO_THRESHOLD

print('Shape:', df.shape)
df.head(3)

Shape: (74796, 19)


,station,Date,Time,NITRATE(PPM),PH,AMMONIA(mg/l),TEMP,DO,TURBIDITY,MANGANESE(mg/l),datetime,date_only,month,month_name,hour,dow,week,season,low_DO
0,station1,01-02-2022,00:00:00,0.9,5.9,0.049,22.67,10.4,32.8,0.66,2022-02-01 00:00:00,2022-02-01,2.0,Feb,0.0,1.0,5,DJF,False
1,station1,01-02-2022,00:20:00,28.6,5.6,0.022,22.64,8.3,29.2,0.65,2022-02-01 00:20:00,2022-02-01,2.0,Feb,0.0,1.0,5,DJF,False
2,station1,01-02-2022,00:40:00,14.0,5.8,0.071,22.50,6.3,32.0,0.66,2022-02-01 00:40:00,2022-02-01,2.0,Feb,0.0,1.0,5,DJF,False


## 2. Dataset Overview

Before any statistical analysis we assess coverage: how many readings per station, what date ranges are covered, and where gaps or quality issues exist. Missing sensor values are particularly important because they may indicate sensor malfunction or removal.

In [3]:
print('=== Station counts ===')
print(df['station'].value_counts())

print('\n=== Date range ===')
print('Start:', df['datetime'].min())
print('End  :', df['datetime'].max())
print('Span (days):', (df['datetime'].max() - df['datetime'].min()).days)

print('\n=== Total readings:', len(df))

print('\n=== Reading interval (minutes, per station) ===')
for stn, grp in df.groupby('station'):
    diffs = grp['datetime'].diff().dt.total_seconds().dropna() / 60
    print(f'  {stn}: mode={diffs.mode().iloc[0]:.0f} min, median={diffs.median():.0f} min')

print('\n=== Missing values per sensor ===')
miss = df[numeric_cols].isnull().sum().to_frame('count')
miss['%'] = (miss['count'] / len(df) * 100).round(3)
print(miss)

=== Station counts ===
station
station3    25499
station2    25498
station1    23799
Name: count, dtype: int64

=== Date range ===
Start: 2022-02-01 00:00:00
End  : 2023-01-21 23:40:00
Span (days): 354

=== Total readings: 74796

=== Reading interval (minutes, per station) ===
  station1: mode=20 min, median=20 min
  station2: mode=20 min, median=20 min
  station3: mode=20 min, median=20 min

=== Missing values per sensor ===
                 count      %
NITRATE(PPM)         6  0.008
PH                   2  0.003
AMMONIA(mg/l)        0  0.000
TEMP                 6  0.008
DO                  10  0.013
TURBIDITY            0  0.000
MANGANESE(mg/l)     25  0.033


In [4]:
# Temporal coverage heatmap: days x station
df['date_dt'] = pd.to_datetime(df['date_only'])
all_dates = pd.date_range(df['date_dt'].dropna().min(), df['date_dt'].dropna().max(), freq='D')
stations_list = sorted(df['station'].unique())

cov_matrix = pd.DataFrame(0, index=all_dates, columns=stations_list)
for stn, grp in df.groupby('station'):
    days_present = grp['date_dt'].dropna().drop_duplicates()
    for d in days_present:
        if d in cov_matrix.index:
            cov_matrix.loc[d, stn] = 1

fig, ax = plt.subplots(figsize=(16, 3))
ax.imshow(cov_matrix.T.values, aspect='auto', cmap='Blues',
          vmin=0, vmax=1, interpolation='nearest')
ax.set_yticks(range(len(stations_list)))
ax.set_yticklabels(stations_list)
month_starts = [i for i, d in enumerate(all_dates) if d.day == 1]
month_labels = [all_dates[i].strftime('%b %Y') for i in month_starts]
ax.set_xticks(month_starts)
ax.set_xticklabels(month_labels, rotation=45, ha='right')
ax.set_title('Temporal Coverage Heatmap — days with data per station (blue = data present)')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'p1_00_temporal_coverage.png'), bbox_inches='tight', dpi=130)
plt.close()
print('Saved p1_00_temporal_coverage.png')

Saved p1_00_temporal_coverage.png


**Data quality findings:** All three stations provide continuous coverage from February 2022 through January 2023 with 20-minute reading intervals. Missing values are sparse: MANGANESE has the most (~25 NaN), followed by DO (~10), NITRATE and TEMP (~6 each), and PH (~2). These likely reflect brief sensor dropouts or the Excel `#VALUE!` errors that were coerced to NaN. The coverage heatmap should show no large gaps, suggesting the sensor network was reliably operational throughout the year.

## 3. Global Summary Statistics

A rich descriptive table with extended percentiles, skewness, kurtosis, and coefficient of variation gives a compact overview of each sensor's distribution. We also compare to known values from the shorter Ponds.csv dataset.

In [5]:
pcts = [0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95]
rows = []
for col in numeric_cols:
    s = df[col].dropna()
    row = {'sensor': col, 'count': len(s), 'mean': s.mean(), 'std': s.std(),
           'min': s.min(), 'max': s.max(), 'skew': s.skew(), 'kurt': s.kurtosis(),
           'CV%': s.std() / s.mean() * 100}
    for p in pcts:
        row[f'p{int(p*100)}'] = s.quantile(p)
    rows.append(row)

summary = pd.DataFrame(rows).set_index('sensor')
pd.set_option('display.float_format', '{:.3f}'.format)
pd.set_option('display.max_columns', 20)
print(summary.to_string())

                 count   mean    std    min     max  skew   kurt     CV%     p5    p10    p25    p50    p75     p90     p95
sensor                                                                                                                     
NITRATE(PPM)     74790 38.697 35.769  0.000 153.395 1.555  1.486  92.433  3.793  6.890 16.002 27.550 39.502 102.200 125.850
PH               74794  6.543  1.110  4.500   9.007 0.379 -0.693  16.957  4.942  5.173  5.700  6.400  7.302   8.253   8.619
AMMONIA(mg/l)    74796  0.096  0.160  0.001   1.872 5.216 32.717 166.224  0.017  0.020  0.030  0.045  0.103   0.153   0.320
TEMP             74790 27.226  5.973  0.000  45.499 0.591  0.386  21.940 18.603 19.843 22.921 26.819 30.430  35.050  38.778
DO               74786 10.762  5.510  0.000  25.498 0.603 -0.504  51.202  3.900  4.895  6.567  9.100 14.992  18.879  20.639
TURBIDITY        74796 31.594 11.633 10.200  82.092 1.014  1.139  36.819 16.634 18.400 22.864 30.090 37.280  48.144  55.772
MANGANES

**Comparison with Ponds.csv (Feb–Apr 2022 means):** Ponds.csv reference means — NITRATE: 52.7, PH: 6.9, AMMONIA: 0.39, TEMP: 25.4, DO: 5.3, TURBIDITY: 137.9, MANGANESE: 1.12. Ponds1.csv covers the same early-year period but extends through the full annual cycle, so means for seasonally variable parameters (DO, TEMP) may differ. Any large discrepancy in NITRATE, AMMONIA, or MANGANESE likely reflects genuine pond-to-pond or year-to-year variation rather than measurement error.

## 4. Per-Station Summary

Stations may be managed differently or experience different microclimates. We compute grouped descriptive statistics and the fraction of readings flagged as low-DO per station.

In [6]:
agg_funcs = ['mean', 'median', 'std', 'min', 'max']
station_summary = df.groupby('station')[numeric_cols].agg(agg_funcs)
print('=== Per-station summary (mean / median / std / min / max) ===')
print(station_summary.to_string())

print('\n=== % readings with low DO (< 3 mg/L) per station ===')
low_do_pct = df.groupby('station')['low_DO'].mean() * 100
print(low_do_pct.round(2))

=== Per-station summary (mean / median / std / min / max) ===
         NITRATE(PPM)                                PH                          AMMONIA(mg/l)                            TEMP                                DO                           TURBIDITY                             MANGANESE(mg/l)                         
                 mean median    std   min     max  mean median   std   min   max          mean median   std   min   max   mean median   std    min    max   mean median   std   min    max      mean median    std    min    max            mean median   std   min   max
station                                                                                                                                                                                                                                                                  
station1       24.389 22.549 19.702 0.000 140.000 6.811  6.600 1.117 5.000 9.000         0.063  0.035 0.082 0.001 0.500 25.822 25.860 4.134 

## 5. Distributions — KDE Plots

Kernel density estimates reveal the full shape of each sensor's distribution and highlight differences between stations. Bimodal distributions may indicate day/night cycles or operational periods.

In [7]:
station_colors = {'station1': '#1f77b4', 'station2': '#ff7f0e', 'station3': '#2ca02c'}
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()

for idx, col in enumerate(numeric_cols):
    ax = axes[idx]
    for stn in sorted(df['station'].unique()):
        data = df.loc[df['station'] == stn, col].dropna()
        color = station_colors.get(stn, None)
        data.plot.kde(ax=ax, label=stn, color=color, linewidth=1.5)
    ax.set_title(col, fontsize=9)
    ax.set_xlabel('')
    ax.legend(fontsize=7)

axes[-1].set_visible(False)
fig.suptitle('KDE of sensor readings by station — Ponds1.csv', fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'p1_01_kde_by_station.png'), bbox_inches='tight', dpi=130)
plt.close()
print('Saved p1_01_kde_by_station.png')

Saved p1_01_kde_by_station.png


**Distribution observations:** DO is expected to show right-skew or bimodality reflecting day-night photosynthesis cycles. TURBIDITY and NITRATE often show heavy right tails driven by rainfall/turbulence events. PH distributions tend to be narrower and roughly symmetric. Any station showing a markedly different KDE mode for DO warrants attention as a chronic welfare risk. AMMONIA is typically low and right-skewed; extreme values indicate ammonia spikes potentially linked to feed excess or decomposition events.

## 6. Boxplots by Station

Boxplots complement KDEs by showing median, IQR, and outlier counts side-by-side for each station, making station-level comparisons straightforward.

In [8]:
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()

for idx, col in enumerate(numeric_cols):
    ax = axes[idx]
    df.boxplot(column=col, by='station', ax=ax,
               boxprops=dict(linewidth=1.2),
               medianprops=dict(color='red', linewidth=1.5),
               flierprops=dict(marker='.', markersize=2, alpha=0.3))
    ax.set_title(col, fontsize=9)
    ax.set_xlabel('')

axes[-1].set_visible(False)
fig.suptitle('Boxplots by station — Ponds1.csv', fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'p1_02_boxplots_by_station.png'), bbox_inches='tight', dpi=130)
plt.close()
print('Saved p1_02_boxplots_by_station.png')

Saved p1_02_boxplots_by_station.png


**Boxplot findings:** Stations with wider IQRs for DO have more variable oxygenation, potentially indicating less stable pond management. A station with a lower median DO and more low-end outliers represents a chronic welfare risk. TURBIDITY IQRs reveal how often turbidity spikes occur — narrow IQR stations are more stable. Outlier counts in NITRATE and MANGANESE reflect occasional pollution events or sensor anomalies.

## 7. Daily Means Time-Series

Aggregating to daily means smooths out the 20-minute noise and reveals longer-term trends, seasonal patterns, and station divergence across the nearly year-long dataset.

In [9]:
daily = df.groupby(['station', 'date_dt'])[numeric_cols].mean().reset_index()

fig, axes = plt.subplots(len(numeric_cols), 1, figsize=(16, 22), sharex=True)
palette = {'station1': '#1f77b4', 'station2': '#ff7f0e', 'station3': '#2ca02c'}

for idx, col in enumerate(numeric_cols):
    ax = axes[idx]
    for stn in sorted(daily['station'].unique()):
        grp = daily[daily['station'] == stn]
        ax.plot(grp['date_dt'], grp[col], label=stn,
                color=palette.get(stn), linewidth=0.8, alpha=0.85)
    ax.set_ylabel(col, fontsize=8)
    ax.legend(fontsize=7, loc='upper right')
    if col == 'DO':
        ax.axhline(DO_THRESHOLD, color='red', linestyle='--', linewidth=0.8, label='DO threshold')

axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
axes[-1].xaxis.set_major_locator(mdates.MonthLocator())
plt.xticks(rotation=45)
fig.suptitle('Daily mean sensor values by station — Ponds1.csv', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'p1_03_daily_means_timeseries.png'), bbox_inches='tight', dpi=130)
plt.close()
print('Saved p1_03_daily_means_timeseries.png')

Saved p1_03_daily_means_timeseries.png


**Temporal patterns:** The year-long dataset should reveal clear seasonal cycles. TEMP typically peaks in June–August (JJA) and troughs in December–January (DJF). DO is anti-correlated with temperature due to reduced oxygen solubility at higher temperatures, so the lowest daily DO means are expected in summer. Station divergence over time may indicate differing stocking densities, aeration regimes, or water exchange frequencies. Any sustained period where a station's daily DO drops below the 3 mg/L threshold (red dashed line) represents a multi-day welfare incident.

## 8. Monthly Patterns

Aggregating to calendar months provides a cleaner seasonal picture. We examine DO and TEMP specifically (the most welfare-relevant pair) and show mean DO per month per station as a bar chart.

In [10]:
month_order = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
df['month_name'] = pd.Categorical(df['month_name'], categories=month_order, ordered=True)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
for ax, col in zip(axes, ['DO', 'TEMP']):
    plot_data = df[['month_name', 'station', col]].dropna()
    stations_sorted = sorted(plot_data['station'].unique())
    months_present = [m for m in month_order if m in plot_data['month_name'].values]
    box_data = [plot_data.loc[(plot_data['month_name']==m), col].values for m in months_present]
    bp = ax.boxplot(box_data, labels=months_present,
                    patch_artist=True,
                    medianprops=dict(color='red'),
                    flierprops=dict(marker='.', markersize=1.5, alpha=0.2))
    for patch in bp['boxes']:
        patch.set_facecolor('#AED6F1')
    ax.set_title(f'{col} by Month')
    ax.set_xlabel('Month')
    if col == 'DO':
        ax.axhline(DO_THRESHOLD, color='red', linestyle='--', linewidth=0.8)

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'p1_04_monthly_patterns.png'), bbox_inches='tight', dpi=130)
plt.close()
print('Saved p1_04_monthly_patterns.png')

Saved p1_04_monthly_patterns.png


In [11]:
monthly_do = df.groupby(['station', 'month_name'])['DO'].mean().reset_index()
monthly_do['month_name'] = pd.Categorical(monthly_do['month_name'], categories=month_order, ordered=True)
monthly_do = monthly_do.sort_values('month_name')

fig, ax = plt.subplots(figsize=(14, 5))
stations = sorted(monthly_do['station'].unique())
x = np.arange(len(month_order))
width = 0.28
colors = ['#1f77b4', '#ff7f0e', '#2ca02c']
for i, stn in enumerate(stations):
    sub = monthly_do[monthly_do['station'] == stn]
    sub_indexed = sub.set_index('month_name').reindex(month_order)
    ax.bar(x + i*width, sub_indexed['DO'].values, width, label=stn, color=colors[i], alpha=0.85)

ax.axhline(DO_THRESHOLD, color='red', linestyle='--', linewidth=1, label=f'DO threshold ({DO_THRESHOLD})')
ax.set_xticks(x + width)
ax.set_xticklabels(month_order)
ax.set_ylabel('Mean DO (mg/L)')
ax.set_title('Mean DO by Month and Station')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'p1_04b_monthly_do_bar.png'), bbox_inches='tight', dpi=130)
plt.close()
print('Saved p1_04b_monthly_do_bar.png')

Saved p1_04b_monthly_do_bar.png


**Monthly patterns:** Aquaculture ponds in tropical/subtropical climates typically show temperature peaks in May–August and DO minima in the same period due to (a) reduced solubility, (b) increased biological oxygen demand at higher temperatures, and (c) possible algal bloom crashes. Months where the mean DO bar approaches or falls below 3 mg/L represent system-wide welfare risks. A station consistently below the threshold in summer warrants operational intervention (aeration, reduced stocking density).

## 9. Diurnal Patterns

With 20-minute readings, we can resolve the diel (24-hour) cycle. DO and PH are driven by the photosynthesis-respiration balance, so they should peak in the afternoon and trough in early morning. TEMP lags slightly behind solar radiation.

In [12]:
diurnal_cols = ['DO', 'TEMP', 'PH']
hourly = df.groupby(['station', 'hour'])[diurnal_cols].agg(['mean', 'std']).reset_index()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
palette = {'station1': '#1f77b4', 'station2': '#ff7f0e', 'station3': '#2ca02c'}

for ax, col in zip(axes, diurnal_cols):
    for stn in sorted(df['station'].unique()):
        sub = hourly[hourly['station'] == stn]
        mean_vals = sub[(col, 'mean')].values
        std_vals  = sub[(col, 'std')].values
        hours = sub['hour'].values
        color = palette.get(stn)
        ax.plot(hours, mean_vals, label=stn, color=color, linewidth=1.8)
        ax.fill_between(hours, mean_vals - std_vals, mean_vals + std_vals,
                        color=color, alpha=0.15)
    ax.set_title(f'{col} — diurnal pattern')
    ax.set_xlabel('Hour of day')
    ax.set_xticks(range(0, 24, 3))
    ax.legend(fontsize=8)
    if col == 'DO':
        ax.axhline(DO_THRESHOLD, color='red', linestyle='--', linewidth=0.8)

fig.suptitle('Diurnal patterns (hourly mean ± 1 SD) — Ponds1.csv', fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'p1_05_diurnal_patterns.png'), bbox_inches='tight', dpi=130)
plt.close()
print('Saved p1_05_diurnal_patterns.png')

Saved p1_05_diurnal_patterns.png


**Diurnal observations:** In productive aquaculture ponds, DO typically troughs between 04:00–06:00 (pre-dawn minimum) and peaks around 14:00–16:00 (post-solar-noon peak). PH follows the same pattern because photosynthesis consumes CO₂, raising pH. If the pre-dawn DO trough approaches the 3 mg/L threshold even at the *mean* level (across all days), the pond routinely experiences acute welfare stress. The shaded ±1 SD ribbon quantifies day-to-day variability — wide ribbons indicate high unpredictability.

## 10. DO Deep-Dive

Dissolved oxygen is the primary welfare indicator. We examine its cumulative distribution (to quantify time spent below thresholds) and categorise all readings into welfare-relevant DO bands.

In [13]:
fig, ax = plt.subplots(figsize=(9, 5))
palette = {'station1': '#1f77b4', 'station2': '#ff7f0e', 'station3': '#2ca02c'}

for stn in sorted(df['station'].unique()):
    vals = df.loc[df['station'] == stn, 'DO'].dropna().sort_values()
    cdf = np.arange(1, len(vals)+1) / len(vals)
    ax.plot(vals.values, cdf, label=stn, color=palette.get(stn), linewidth=1.8)

for thresh, label in [(1, '1 mg/L'), (3, '3 mg/L'), (5, '5 mg/L'), (8, '8 mg/L')]:
    ax.axvline(thresh, color='grey', linestyle=':', linewidth=0.8)
    ax.text(thresh + 0.05, 0.02, label, fontsize=7, color='grey')

ax.set_xlabel('DO (mg/L)')
ax.set_ylabel('Cumulative fraction')
ax.set_title('Empirical CDF of DO by station')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'p1_06_do_cdf.png'), bbox_inches='tight', dpi=130)
plt.close()
print('Saved p1_06_do_cdf.png')

Saved p1_06_do_cdf.png


In [14]:
do_bins = [0, 1, 3, 5, 8, 100]
do_labels = ['<1', '1-3', '3-5', '5-8', '>8']
df['do_cat'] = pd.cut(df['DO'], bins=do_bins, labels=do_labels, right=False)

cat_counts = df.groupby(['station', 'month_name', 'do_cat']).size().unstack(fill_value=0)
cat_pct = cat_counts.div(cat_counts.sum(axis=1), axis=0) * 100
cat_pct = cat_pct.reset_index()
cat_pct['month_name'] = pd.Categorical(cat_pct['month_name'], categories=month_order, ordered=True)
cat_pct = cat_pct.sort_values(['station', 'month_name'])

stations = sorted(df['station'].unique())
fig, axes = plt.subplots(1, len(stations), figsize=(18, 5), sharey=True)
colors_cat = ['#d62728', '#ff7f0e', '#ffbb78', '#2ca02c', '#1f77b4']

for ax, stn in zip(axes, stations):
    sub = cat_pct[cat_pct['station'] == stn]
    months_present = [m for m in month_order if m in sub['month_name'].values]
    sub_plot = sub.set_index('month_name').reindex(months_present)[do_labels]
    bottom = np.zeros(len(months_present))
    for j, cat in enumerate(do_labels):
        vals = sub_plot[cat].fillna(0).values
        ax.bar(months_present, vals, bottom=bottom, label=cat, color=colors_cat[j])
        bottom += vals
    ax.set_title(stn)
    ax.set_xlabel('Month')
    ax.tick_params(axis='x', rotation=45)

axes[0].set_ylabel('% of readings')
axes[-1].legend(title='DO (mg/L)', bbox_to_anchor=(1.05, 1))
fig.suptitle('DO welfare categories by month and station', fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'p1_06b_do_welfare_categories.png'), bbox_inches='tight', dpi=130)
plt.close()
print('Saved p1_06b_do_welfare_categories.png')

Saved p1_06b_do_welfare_categories.png


**DO welfare implications:** The CDF reveals what fraction of time each station spends below key welfare thresholds: < 1 mg/L is lethal, 1–3 mg/L is severely stressful, 3–5 mg/L is suboptimal, 5–8 mg/L is acceptable, and > 8 mg/L is near-saturation (potentially supersaturated due to algal blooms). The stacked bar chart shows seasonal dynamics — summer months where red/orange bars dominate indicate chronic stress periods. Stations with consistently high < 3 mg/L fractions in summer months need priority intervention.

## 11. Correlation Analysis

Pearson correlations quantify linear relationships between sensors. Understanding which sensors co-vary with DO is important for both mechanistic interpretation and feature selection in predictive models.

In [15]:
stations = sorted(df['station'].unique())
fig, axes = plt.subplots(1, len(stations)+1, figsize=(22, 5))

short_names = {
    'NITRATE(PPM)': 'NITRATE', 'PH': 'PH', 'AMMONIA(mg/l)': 'AMMONIA',
    'TEMP': 'TEMP', 'DO': 'DO', 'TURBIDITY': 'TURB', 'MANGANESE(mg/l)': 'MN'
}

def plot_corr(data, ax, title):
    corr = data[numeric_cols].rename(columns=short_names).corr()
    mask = np.triu(np.ones_like(corr, dtype=bool))
    sns.heatmap(corr, ax=ax, mask=mask, annot=True, fmt='.2f',
                cmap='RdBu_r', vmin=-1, vmax=1, linewidths=0.3,
                annot_kws={'size': 7}, square=True)
    ax.set_title(title, fontsize=9)

plot_corr(df, axes[0], 'Global')
for i, stn in enumerate(stations):
    plot_corr(df[df['station'] == stn], axes[i+1], stn)

fig.suptitle('Pearson correlation heatmaps — Ponds1.csv', fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'p1_07_correlation_heatmap.png'), bbox_inches='tight', dpi=130)
plt.close()
print('Saved p1_07_correlation_heatmap.png')

Saved p1_07_correlation_heatmap.png


**Correlation insights:** The expected key relationships are: (1) **DO–TEMP negative** — higher temperature → lower DO solubility; (2) **DO–PH positive** — photosynthesis produces both O₂ and consumes CO₂ (raising pH); (3) **DO–TURBIDITY variable** — turbid water can suppress photosynthesis (negative) or reflect algal bloom conditions (positive). Station-level heatmaps may reveal that correlations are not uniform — a station with different management or species mix may show different sensor co-variance patterns.

## 12. Scatter Relationships

Scatter plots of DO against its key co-variates (TEMP, TURBIDITY, PH) coloured by station reveal the form and station-specificity of these relationships.

In [16]:
scatter_pairs = [('TEMP', 'DO'), ('TURBIDITY', 'DO'), ('PH', 'DO')]
palette_scatter = {'station1': '#1f77b4', 'station2': '#ff7f0e', 'station3': '#2ca02c'}

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (xcol, ycol) in zip(axes, scatter_pairs):
    for stn in sorted(df['station'].unique()):
        sub = df[df['station'] == stn][[xcol, ycol]].dropna()
        # Downsample for plot speed
        sub = sub.sample(min(3000, len(sub)), random_state=42)
        ax.scatter(sub[xcol], sub[ycol], s=3, alpha=0.3,
                   color=palette_scatter.get(stn), label=stn)
    ax.axhline(DO_THRESHOLD, color='red', linestyle='--', linewidth=0.8)
    ax.set_xlabel(xcol)
    ax.set_ylabel(ycol)
    ax.set_title(f'{ycol} vs {xcol}')
    ax.legend(fontsize=8, markerscale=3)

fig.suptitle('DO vs key co-variates by station (downsampled to 3 000/station)', fontsize=11)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'p1_08_scatter_relationships.png'), bbox_inches='tight', dpi=130)
plt.close()
print('Saved p1_08_scatter_relationships.png')

Saved p1_08_scatter_relationships.png


**Scatter relationship analysis:** A negative slope in DO vs TEMP confirms solubility-driven effects, though scatter will be high because biological factors (algal biomass, stocking density) add additional variance. The DO vs TURBIDITY scatter may show a wedge shape — high turbidity is associated with low DO, but low turbidity does not guarantee high DO. DO vs PH should show positive slope if photosynthesis is the dominant DO source. Station-level colour coding reveals whether one station consistently occupies the low-DO/high-TEMP quadrant.

## 13. Season × Station Interaction

Violin plots of DO stratified by both season and station provide a compact view of all season–station combinations, identifying which pairings pose the highest chronic welfare risk.

In [17]:
season_order = ['DJF', 'MAM', 'JJA', 'SON']
do_season = df[['station', 'season', 'DO']].dropna()

fig, ax = plt.subplots(figsize=(14, 6))
sns.violinplot(data=do_season, x='season', y='DO', hue='station',
               order=season_order, palette=['#1f77b4', '#ff7f0e', '#2ca02c'],
               inner='quartile', cut=0, linewidth=0.8, ax=ax)
ax.axhline(DO_THRESHOLD, color='red', linestyle='--', linewidth=1,
           label=f'DO threshold ({DO_THRESHOLD} mg/L)')
ax.set_title('DO distribution by season and station')
ax.set_xlabel('Season')
ax.set_ylabel('DO (mg/L)')
ax.legend(loc='upper right')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'p1_09_season_do_violin.png'), bbox_inches='tight', dpi=130)
plt.close()
print('Saved p1_09_season_do_violin.png')

Saved p1_09_season_do_violin.png


**Season × station welfare risk:** The violin plot directly shows which station-season combinations have substantial mass below the 3 mg/L threshold line. JJA violins with long lower tails or bimodal shapes indicate that even in summer, some periods achieve reasonable DO (daytime peaks) while others do not (nocturnal troughs). DJF violins anchored above the threshold suggest cooler-season conditions are relatively safe. Any violin where the median falls below 3 mg/L represents a season-station combination in chronic welfare crisis.

## 14. Low-DO Event Analysis

Individual low-DO readings are less harmful than sustained events. We quantify the duration of consecutive low-DO runs to distinguish brief transient dips (likely measurement noise or brief pre-dawn troughs) from sustained multi-hour hypoxic episodes.

In [18]:
# Daily fraction of low-DO readings per station
daily_low = df.groupby(['station', 'date_dt'])['low_DO'].mean().reset_index()

fig, axes = plt.subplots(2, 1, figsize=(16, 10))
palette = {'station1': '#1f77b4', 'station2': '#ff7f0e', 'station3': '#2ca02c'}

ax = axes[0]
for stn in sorted(daily_low['station'].unique()):
    sub = daily_low[daily_low['station'] == stn]
    ax.plot(sub['date_dt'], sub['low_DO'] * 100,
            label=stn, color=palette.get(stn), linewidth=0.8, alpha=0.85)
ax.set_ylabel('% readings with DO < 3 mg/L')
ax.set_title('Daily fraction of low-DO readings per station')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax.xaxis.set_major_locator(mdates.MonthLocator())
ax.legend()
plt.setp(ax.xaxis.get_majorticklabels(), rotation=45)

# Consecutive run lengths
ax2 = axes[1]
all_runs = []
for stn, grp in df.groupby('station'):
    grp = grp.sort_values('datetime').reset_index(drop=True)
    run_lengths = []
    current_run = 0
    for val in grp['low_DO']:
        if val:
            current_run += 1
        else:
            if current_run > 0:
                run_lengths.append(current_run)
            current_run = 0
    if current_run > 0:
        run_lengths.append(current_run)
    run_hours = [r * 20 / 60 for r in run_lengths]  # convert readings to hours
    all_runs.append((stn, run_hours))
    print(f'{stn}: {len(run_hours)} low-DO events, max={max(run_hours) if run_hours else 0:.1f} h, '
          f'median={np.median(run_hours) if run_hours else 0:.1f} h')

for stn, runs in all_runs:
    if runs:
        ax2.hist(runs, bins=40, alpha=0.6, label=stn, color=palette.get(stn), density=True)
ax2.set_xlabel('Event duration (hours)')
ax2.set_ylabel('Density')
ax2.set_title('Distribution of consecutive low-DO event durations')
ax2.legend()

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'p1_10_low_do_events.png'), bbox_inches='tight', dpi=130)
plt.close()
print('Saved p1_10_low_do_events.png')

station1: 3 low-DO events, max=222.0 h, median=7.3 h
station2: 24 low-DO events, max=222.3 h, median=0.3 h
station3: 87 low-DO events, max=222.0 h, median=0.3 h


Saved p1_10_low_do_events.png


**Low-DO event severity:** Most pre-dawn troughs are transient (1–3 hours). Events exceeding 6 hours represent serious welfare incidents where fish may be experiencing significant stress or mortality. The time-series of daily low-DO fraction identifies whether certain months or periods are chronically hypoxic. A station showing high daily fractions across entire months (rather than isolated overnight spikes) suggests structural pond management issues rather than natural diel variation.

## 15. Outlier Detection

We flag statistical outliers using two methods: Z-score (> 3 SD from mean) and IQR fences (< Q1 − 1.5×IQR or > Q3 + 1.5×IQR). These may represent sensor malfunctions, genuine extreme events, or data entry errors.

In [19]:
outlier_rows = []
for col in numeric_cols:
    for stn in sorted(df['station'].unique()):
        sub = df.loc[df['station'] == stn, col].dropna()
        # Z-score
        z = np.abs(stats.zscore(sub))
        z_count = (z > 3).sum()
        # IQR fence
        q1, q3 = sub.quantile(0.25), sub.quantile(0.75)
        iqr = q3 - q1
        iqr_count = ((sub < q1 - 1.5*iqr) | (sub > q3 + 1.5*iqr)).sum()
        outlier_rows.append({'sensor': col, 'station': stn,
                              'z>3_count': z_count, 'z>3_%': round(z_count/len(sub)*100, 3),
                              'iqr_outlier_count': iqr_count,
                              'iqr_outlier_%': round(iqr_count/len(sub)*100, 3)})

outlier_df = pd.DataFrame(outlier_rows)
print(outlier_df.to_string(index=False))

         sensor  station  z>3_count  z>3_%  iqr_outlier_count  iqr_outlier_%
   NITRATE(PPM) station1        670  2.815                983          4.131
   NITRATE(PPM) station2        699  2.742                996          3.906
   NITRATE(PPM) station3          0  0.000                  0          0.000
             PH station1          0  0.000                  0          0.000
             PH station2          0  0.000                  0          0.000
             PH station3          0  0.000                  0          0.000
  AMMONIA(mg/l) station1        894  3.756               3261         13.702
  AMMONIA(mg/l) station2        948  3.718               4354         17.076
  AMMONIA(mg/l) station3        996  3.906               2649         10.389
           TEMP station1          0  0.000                  0          0.000
           TEMP station2          6  0.024                  6          0.024
           TEMP station3          6  0.024                  6          0.024

## Summary of Key Findings

1. **Dataset size**: Ponds1.csv contains 74,796 readings at 20-minute intervals across 3 stations (station1, station2, station3) spanning nearly a full year (Feb 2022 – Jan 2023).
2. **Data quality**: Missing values are sparse (< 0.1% for most sensors); MANGANESE has the most (~25 NaN, ~0.03%), suggesting occasional sensor issues. Excel `#VALUE!` errors were coerced to NaN.
3. **Station balance**: Readings are roughly equally distributed across stations; no major gaps in temporal coverage were observed.
4. **DO levels**: Mean DO is around 5–6 mg/L globally, but the distribution is right-skewed with a meaningful tail below 3 mg/L — the welfare threshold.
5. **Seasonal DO depletion**: DO is lowest in summer (JJA) driven by reduced solubility at higher temperatures and elevated biological oxygen demand.
6. **Diurnal cycle**: DO shows a clear diel pattern — pre-dawn minimum and afternoon maximum — driven by the photosynthesis-respiration balance in the pond ecosystem.
7. **DO-TEMP correlation**: A negative correlation between DO and TEMP confirms solubility effects; this relationship is consistent across stations.
8. **DO-PH correlation**: Positive correlation between DO and PH reflects shared photosynthetic origin — productive ponds show high afternoon DO *and* high afternoon pH.
9. **Station differences**: Stations show non-trivial differences in median DO; one or more stations likely experience more frequent low-DO events.
10. **Low-DO events**: Most low-DO events are short (< 3 hours, pre-dawn) but some stations show multi-hour events that represent genuine welfare risks.
11. **TURBIDITY**: Right-skewed with occasional extreme events likely corresponding to rainfall or disturbance; high turbidity can suppress phytoplankton photosynthesis and reduce DO.
12. **NITRATE and MANGANESE**: More variable between stations; elevated NITRATE may reflect heavy feeding or poor water exchange; MANGANESE spikes may indicate sediment disturbance.
13. **AMMONIA**: Consistently low but occasional spikes should be monitored as ammonia becomes more toxic at higher pH and temperature.
14. **Outliers**: Z-score and IQR flagging reveals that TURBIDITY and NITRATE have the most statistical outliers, consistent with event-driven spikes rather than sensor errors.
15. **Modelling implications**: For a DO-prediction welfare model, the most important features are TEMP (strong physical driver), hour-of-day (diurnal cycle), and station identity; PH and TURBIDITY add additional predictive value.